In [ ]:
# hide
import numpy as np
import matplotlib.pyplot as plt
import pyquist as pq

PLAYBACK_SR = 44100


def midi_to_hz(m):
    return 440.0 * 2.0 ** ((m - 69) / 12.0)


def aliased(f, f_s):
    m = np.mod(f, f_s)
    return np.minimum(m, f_s - m)


def show_aliasing(f_s, control_points):
    # control_points: list of (time_seconds, midi_note)
    times = [p[0] for p in control_points]
    notes = [p[1] for p in control_points]
    dur = times[-1]

    # true frequency contour and the frequency actually heard after sampling
    n = np.arange(int(dur * f_s))
    freq = midi_to_hz(np.interp(n / f_s, times, notes))

    # synthesize at f_s by accumulating phase (Chapter 6), resample for playback
    x = np.sin(np.cumsum(2 * np.pi * freq / f_s))
    audio = pq.Audio(x.astype(np.float32), int(f_s)).resample(PLAYBACK_SR)

    t = n / f_s
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.plot(t, freq, color="#007BC0", label="true frequency")
    ax.plot(t, aliased(freq, f_s), color="#FDB515", ls="--", label="heard (aliased)")
    ax.axhline(f_s / 2, color="#C41230", lw=1.3, label="Nyquist $f_s/2$")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (Hz)")
    ax.set_ylim(0, max(freq.max(), f_s / 2) * 1.15)
    ax.legend(loc="upper right", fontsize=9)
    plt.show()
    return pq.play(audio)


In [ ]:
# Edit the sample rate and the pitch contour, then run to see and hear aliasing.
# Each control point is (time in seconds, MIDI note). Note 69 = A4 = 440 Hz.
f_s = 500  # sampling rate (Hz): lower it to force aliasing

pitch_contour = [
    (0.0, 57),   # A3, 220 Hz
    (1.0, 57),
    (6.0, 81),   # A5, 880 Hz
    (7.0, 81),
    (12.0, 57),
    (13.0, 57),
]

show_aliasing(f_s, pitch_contour)
